# Spotify Music Recommendation System
## Notebook 05 — Content-Based Recommender

**Purpose:** Use cosine similarity on the 14-dimensional feature vector to recommend the most musically similar songs for any given track.

## Table of Contents
1. [Setup & Imports](#1-setup--imports)
2. [Load Feature Matrix](#2-load-feature-matrix)
3. [What is Cosine Similarity?](#3-what-is-cosine-similarity)
4. [Recommendation Function](#4-recommendation-function)
5. [Test the Recommender](#5-test-the-recommender)
6. [Similarity Score Analysis](#6-similarity-score-analysis)
7. [Save the Model](#7-save-the-model)
8. [Summary](#8-summary)

## 1. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

DATA_DIR  = Path('../data/processed')
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load Feature Matrix

In [ ]:
feature_matrix = pd.read_csv(DATA_DIR / 'feature_matrix.csv').values  # numpy array
track_index    = pd.read_csv(DATA_DIR / 'track_index.csv')

with open(DATA_DIR / 'feature_columns.json') as f:
    feature_columns = json.load(f)

print(f'Feature matrix : {feature_matrix.shape}  (rows=tracks, cols=features)')
print(f'Track index    : {track_index.shape}')
print(f'Features used  : {feature_columns}')
track_index.head(3)

## 3. What is Cosine Similarity?

Each track is represented as a 14-dimensional vector of normalised audio features.

Cosine similarity measures the **angle** between two vectors:
- **1.0** = same direction = very similar audio profile
- **0.0** = perpendicular = nothing in common

To recommend songs for a given track, we compute its similarity to **all other tracks** and return the top-N with the highest scores. With 170K tracks and 14 features, this runs in under 1 second.

In [ ]:
# Quick demo: similarity of a track with itself
v = feature_matrix[0].reshape(1, -1)
print(f'Self-similarity of track 0: {cosine_similarity(v, v)[0, 0]:.4f}  (always 1.0)')

# Similarity between two random tracks
v2 = feature_matrix[999].reshape(1, -1)
print(f'Similarity between track 0 and track 999: {cosine_similarity(v, v2)[0, 0]:.4f}')

## 4. Recommendation Function

In [ ]:
def get_recommendations(
    song_name: str,
    top_n: int = 10,
    exclude_same_artist: bool = False,
    year_range: tuple = None,
) -> pd.DataFrame:
    """
    Return the top-N most similar songs for a given track title.

    Parameters
    ----------
    song_name           : Track title (case-insensitive, partial match allowed).
    top_n               : Number of recommendations.
    exclude_same_artist : If True, removes songs by the same artist.
    year_range          : Optional (start_year, end_year) to filter by release year.
    """
    lower = song_name.lower()

    # Find the song — exact match first, then partial
    exact   = track_index[track_index['name'].str.lower() == lower]
    partial = track_index[track_index['name'].str.lower().str.contains(lower, na=False, regex=False)]
    matches = exact if not exact.empty else partial

    if matches.empty:
        print(f"Song not found: '{song_name}'")
        return pd.DataFrame()

    # If multiple matches, pick the most popular one
    q = matches.loc[matches['popularity'].idxmax()]
    q_idx = q.name
    print(f"Found: '{q['name']}' by {q['artists']} ({int(q['year'])}, pop={int(q['popularity'])})")

    # Compute cosine similarity against all tracks
    q_vec = feature_matrix[q_idx].reshape(1, -1)
    sims  = cosine_similarity(q_vec, feature_matrix).flatten()

    # Build results
    results = track_index.copy()
    results['similarity'] = sims
    results = results[results.index != q_idx]  # remove the query itself

    if exclude_same_artist:
        q_artists = set(str(q['artists']).split(', '))
        results = results[
            ~results['artists'].apply(
                lambda a: bool(set(str(a).split(', ')) & q_artists)
            )
        ]

    if year_range:
        results = results[results['year'].between(year_range[0], year_range[1])]

    top = results.nlargest(top_n, 'similarity').reset_index(drop=True)
    top.index += 1  # 1-based ranking
    return top[['name', 'artists', 'year', 'popularity', 'similarity']]


print('Recommendation function ready.')

## 5. Test the Recommender

In [ ]:
# Basic test
recs = get_recommendations('White Christmas', top_n=10)
print(recs.to_string())

In [ ]:
# Test with filter: post-2000 songs, different artist
print('\n--- Bohemian Rhapsody | post-2000 | different artist ---')
recs = get_recommendations('Bohemian Rhapsody', top_n=8,
                            exclude_same_artist=True, year_range=(2000, 2020))
print(recs.to_string())

In [ ]:
# Test several songs
for song in ['Shape of You', 'Lose Yourself', 'Nothing Else Matters']:
    print(f'\n=== {song} ===')
    recs = get_recommendations(song, top_n=5)
    if not recs.empty:
        print(recs[['name', 'artists', 'year', 'similarity']].to_string())

## 6. Similarity Score Analysis

For a given query song, we look at the full distribution of cosine similarity scores across all 170K tracks. The top recommendations should be well above average.

In [ ]:
sample = 'White Christmas'
lower  = sample.lower()
match  = track_index[track_index['name'].str.lower() == lower]

if not match.empty:
    q_idx = match.loc[match['popularity'].idxmax()].name
    q_vec = feature_matrix[q_idx].reshape(1, -1)
    all_sims = cosine_similarity(q_vec, feature_matrix).flatten()
    all_sims = np.delete(all_sims, q_idx)  # remove self

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(all_sims, bins=60, color='steelblue', edgecolor='none')
    ax.axvline(np.percentile(all_sims, 99), color='red', linewidth=1.5,
               linestyle='--', label=f'Top 1% threshold: {np.percentile(all_sims, 99):.3f}')
    ax.set_title(f'Similarity scores for all tracks vs "{sample}"', fontweight='bold')
    ax.set_xlabel('Cosine Similarity')
    ax.set_ylabel('Number of Tracks')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f'Mean similarity  : {all_sims.mean():.4f}')
    print(f'Top 10 recs are  : {np.sort(all_sims)[-10:][::-1].round(4)}')

## 7. Save the Model

In [ ]:
payload = {
    'feature_matrix' : feature_matrix,
    'track_index'    : track_index,
    'feature_columns': feature_columns,
}

with open(MODEL_DIR / 'recommender_payload.pkl', 'wb') as f:
    pickle.dump(payload, f)

print('Saved: models/recommender_payload.pkl')
print(f'  feature_matrix : {feature_matrix.shape}')
print(f'  track_index    : {track_index.shape}')

## 8. Summary

### How It Works
1. Look up the query song in `track_index` to get its row number.
2. Retrieve the corresponding 14-dimensional vector from `feature_matrix`.
3. Compute cosine similarity against **all other tracks** (~170K dot products).
4. Return the top-N tracks with the highest scores.

### Optional Filters
- `exclude_same_artist=True` — avoids recommending the same artist repeatedly
- `year_range=(start, end)` — limits results to a specific era

### Limitations
- Content-only: ignores listening history or user preferences
- Popularity is included in the feature vector, which can bias results toward recent popular songs

**Next:** `06_evaluation.ipynb` — measure recommendation quality.